# EfficientNet-B0 Food Classifier -- Google Colab Notebook

16-class image food classification with EfficientNet-B0 and two-stage transfer learning.

**Framework:** TensorFlow / Keras  
**Input:** Single image (224x224)  
**Output:** Class probabilities (softmax over 16 classes)

## 1. Setup -- Clone & Install

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/whispr-messenger/moderation-service.git'
REPO_DIR = '/content/moderation-service'
BRANCH = 'WHISPR-668'  # change to 'main' if your branch is not pushed

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f'Git clone failed:\n{result.stderr}')
        print('Falling back to main branch...')
        result = subprocess.run(
            ['git', 'clone', REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'Clone failed: {result.stderr}')
        print('Cloned main branch successfully')
    else:
        print(f'Cloned {BRANCH} successfully')
else:
    print(f'Repository already cloned at {REPO_DIR}')

EFFNET_DIR = os.path.join(REPO_DIR, 'src', 'efficientnet_lite_gpu')
os.chdir(EFFNET_DIR)
if EFFNET_DIR not in sys.path:
    sys.path.insert(0, EFFNET_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Mount Google Drive (optional, for checkpoint persistence)
DRIVE_CHECKPOINT_DIR = None
DRIVE_MODEL_PATH = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/whispr-checkpoints/efficientnet'
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    DRIVE_MODEL_PATH = os.path.join(DRIVE_CHECKPOINT_DIR, 'BestModelEfficientNetLite.h5')
    print(f'Drive checkpoint dir: {DRIVE_CHECKPOINT_DIR}')
    if os.path.exists(DRIVE_MODEL_PATH):
        print(f'Found existing model: {DRIVE_MODEL_PATH}')
    else:
        print('No existing model -- training will start from scratch')
except Exception as e:
    print(f'Drive not mounted ({e}) -- training works but no checkpoint sync')

In [ ]:
# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
print('Dependencies installed')

## 2. Environment Check

In [ ]:
import importlib
from pathlib import Path

print(f'Python {sys.version}')
print()
for pkg in ['tensorflow', 'numpy', 'matplotlib', 'sklearn', 'cv2', 'PIL', 'yaml', 'huggingface_hub']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg:18s} {getattr(m, "__version__", "ok")}')
    except ImportError:
        print(f'  {pkg:18s} NOT INSTALLED')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\nGPU available: {len(gpus) > 0}')
if gpus:
    print(f'GPU: {gpus[0]}')

## 3. HuggingFace Authentication

Add `HF_TOKEN` to **Colab Secrets** (key icon in the left sidebar).
Make sure the token has **write permissions** to your HF namespace.

In [ ]:
HF_TOKEN = ''

# Try Colab Secrets first
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
    if HF_TOKEN:
        print('HF token loaded from Colab secrets')
except Exception:
    pass

# Fallback: env var or .env file
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    env_path = Path(REPO_DIR) / '.env'
    if not HF_TOKEN and env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip().strip('"').strip("'")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN set')
else:
    print('No HF_TOKEN -- public download still works, uploads will be skipped')

## 4. Data -- Paths & Config

In [ ]:
import yaml

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

train_cfg = cfg['train_config']
DATASET_DIR = Path.cwd() / train_cfg['dataset_dir']
TRAIN_DIR = DATASET_DIR / train_cfg['train_dir']
TEST_DIR = DATASET_DIR / train_cfg['test_dir']
RESULTS_DIR = Path.cwd() / train_cfg['results_dir']

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

HEALTH_LABELS = {
    'fruits': 'healthy', 'vegetables': 'healthy', 'salads': 'healthy',
    'seafood': 'healthy', 'grilled_meat': 'healthy', 'grain_bowls': 'healthy',
    'soups': 'healthy', 'smoothies': 'healthy',
    'burgers': 'unhealthy', 'pizza': 'unhealthy', 'fried_food': 'unhealthy',
    'desserts': 'unhealthy', 'candy_sweets': 'unhealthy',
    'salty_snacks': 'unhealthy', 'sugary_drinks': 'unhealthy',
    'not_food': 'not_food',
}

print(f'Dataset dir: {DATASET_DIR}')
print(f'Train dir:   {TRAIN_DIR}')
print(f'Test dir:    {TEST_DIR}')
print(f'Classes:     {len(HEALTH_LABELS)}')

## 5. Dataset Download

**Option A:** Download from HuggingFace (fast, ~30s).  
**Option B:** Scrape from DuckDuckGo if HF fails or repo is empty (~1h).

In [ ]:
HF_DATASET_REPO = 'maia2000/efficientnet-food-dataset'

def _count_images(root):
    if not root.exists():
        return 0
    total = 0
    for cls in root.iterdir():
        if cls.is_dir():
            total += sum(1 for f in cls.rglob('*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
    return total

# Try HuggingFace first
downloaded_from_hf = False
try:
    from huggingface_hub import snapshot_download
    print(f'Trying to download from HuggingFace ({HF_DATASET_REPO})...')
    snapshot_download(
        repo_id=HF_DATASET_REPO,
        repo_type='dataset',
        local_dir=str(DATASET_DIR),
        allow_patterns=['Train/**', 'Test/**'],
        token=HF_TOKEN if HF_TOKEN else None,
    )
    if _count_images(TRAIN_DIR) > 100:
        downloaded_from_hf = True
        print(f'Dataset downloaded from HuggingFace')
    else:
        print('HF download returned no images -- will scrape')
except Exception as e:
    print(f'HF download failed: {e}')
    print('Will scrape from DuckDuckGo instead...')

print(f'\nDownloaded from HF: {downloaded_from_hf}')
print(f'Images in Train dir: {_count_images(TRAIN_DIR)}')

In [ ]:
# Scrape from DuckDuckGo if HF download failed
# Classes with enough images are skipped automatically.

FORCE_SCRAPE = False  # Set True to scrape even if HF download succeeded

if not downloaded_from_hf or FORCE_SCRAPE:
    from tools.fetch_google_dataset import run as fetch_dataset
    print('Scraping images from DuckDuckGo (~1h for full 16 classes)...')
    fetch_dataset(root_dir=Path.cwd(), dry_run=False)
    print('Scraping complete')
else:
    print('Skipping scrape -- HF download succeeded')

## 6. Upload Dataset to HuggingFace

Auto-runs after scraping so teammates can use the HF dataset next time.

In [ ]:
# Upload right after download (skipped automatically if no HF_TOKEN or HF download succeeded)
SKIP_DATASET_UPLOAD = False

if not SKIP_DATASET_UPLOAD and HF_TOKEN and not downloaded_from_hf:
    try:
        from huggingface_hub import HfApi, create_repo
        api = HfApi(token=HF_TOKEN)
        create_repo(HF_DATASET_REPO, repo_type='dataset', token=HF_TOKEN, exist_ok=True)
        print(f'Uploading dataset to {HF_DATASET_REPO}...')
        if TRAIN_DIR.exists() and _count_images(TRAIN_DIR) > 0:
            print('  Uploading Train...')
            api.upload_folder(
                repo_id=HF_DATASET_REPO, repo_type='dataset',
                folder_path=str(TRAIN_DIR), path_in_repo='Train',
                commit_message='Upload training images', token=HF_TOKEN,
            )
        if TEST_DIR.exists() and _count_images(TEST_DIR) > 0:
            print('  Uploading Test...')
            api.upload_folder(
                repo_id=HF_DATASET_REPO, repo_type='dataset',
                folder_path=str(TEST_DIR), path_in_repo='Test',
                commit_message='Upload test images', token=HF_TOKEN,
            )
        print(f'Dataset uploaded: https://huggingface.co/datasets/{HF_DATASET_REPO}')
    except Exception as e:
        print(f'Upload failed: {e}')
elif downloaded_from_hf:
    print('[SKIP] Dataset downloaded from HF -- no need to re-upload')
elif not HF_TOKEN:
    print('[SKIP] No HF_TOKEN -- add it to Colab Secrets to enable upload')
else:
    print('[SKIP] SKIP_DATASET_UPLOAD = True')

## 7. Dataset Inspection

In [ ]:
def count_images(root_dir):
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
    counts = {}
    for class_name in sorted(os.listdir(root_dir)):
        class_path = Path(root_dir) / class_name
        if not class_path.is_dir():
            continue
        n = sum(1 for f in class_path.rglob('*') if f.suffix.lower() in image_exts)
        counts[class_name] = n
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR) if TEST_DIR.exists() else {}

print(f'Train: {sum(train_counts.values())} images across {len(train_counts)} classes')
for cls, n in train_counts.items():
    print(f'  {cls:18s} {n:5d}')
if test_counts:
    print(f'\nTest: {sum(test_counts.values())} images across {len(test_counts)} classes')
    for cls, n in test_counts.items():
        print(f'  {cls:18s} {n:5d}')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img

classes = sorted(os.listdir(TRAIN_DIR))
classes = [c for c in classes if (TRAIN_DIR / c).is_dir()]
n_show = min(len(classes), 6)

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
if n_show == 1:
    axes = axes.reshape(2, 1)

for i, cls in enumerate(classes[:n_show]):
    class_dir = TRAIN_DIR / cls
    imgs = [f for f in class_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    for row in range(min(2, len(imgs))):
        img = load_img(imgs[row], target_size=(224, 224))
        axes[row, i].imshow(img)
        axes[row, i].set_title(cls, fontsize=9)
        axes[row, i].axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
classes_sorted = sorted(train_counts.keys())
counts_sorted = [train_counts[c] for c in classes_sorted]
plt.bar(range(len(classes_sorted)), counts_sorted, color='steelblue')
plt.xticks(range(len(classes_sorted)), classes_sorted, rotation=45, ha='right')
plt.ylabel('Number of Images')
plt.title('Training Set Class Distribution')
plt.tight_layout()
plt.show()

## 8. Training

Two-stage transfer learning: head training + fine-tuning.

In [ ]:
import shutil
from train.train import run as train_run

MODEL_NAME = 'BestModelEfficientNetLite.h5'
MODEL_PATH = Path.cwd() / MODEL_NAME

# Resume from Drive if available
if DRIVE_MODEL_PATH and os.path.exists(DRIVE_MODEL_PATH):
    shutil.copy2(DRIVE_MODEL_PATH, str(MODEL_PATH))
    print(f'Restored model from Drive: {DRIVE_MODEL_PATH}')
else:
    print('Starting training from scratch')

train_config = {
    'train_config': {
        'training_device': 'gpu' if tf.config.list_physical_devices('GPU') else 'cpu',
        'mixed_precision': False,
        'dataset_dir': 'train/dataset',
        'results_dir': 'train/results',
        'data_exploration_dir': 'data_exploration',
        'evaluation_results_dir': 'evaluation_results',
        'training_logs_dir': 'training_logs',
        'training_results_dir': 'training_results',
        'train_dir': 'Train',
        'val_dir': 'Val',
        'test_dir': 'Test',
        'batch_size': 8,
        'image_size': 224,
        'initial_epochs': 50,
        'fine_tune': True,
        'fine_tune_epochs': 50,
        'data_augmentation': {
            'randomFlip': 'horizontal',
            'randomRotation': 0.05,
            'randomZoom': 0.1,
            'randomContrast': 0.1,
        },
        'model_config': {
            'model_name': 'efficientnet-b0',
            'include_top': False,
            'weights': 'imagenet',
            'input_shape': '(224, 224, 3)',
            'trainable': False,
            'optimizer': 'adam',
            'output_activation': 'softmax',
            'learning_rate': 0.01,
            'loss': 'sparse_categorical_crossentropy',
            'metrics': ['accuracy'],
            'EarlyStopping': {'monitor': 'val_loss', 'patience': 8, 'restore_best_weights': True},
            'ReduceLROnPlateau': {'monitor': 'val_loss', 'factor': 0.5, 'patience': 3, 'min_lr': 1e-6},
        },
    },
    'sys_config': {'disable_XLA_logs': True, 'tf_force_gpu_allow_growth': True},
    'compilation_config': {'compiler': 'gcc', 'compiler_args': [], 'model_name': MODEL_NAME},
}

train_run(train_config)

if MODEL_PATH.exists() and DRIVE_MODEL_PATH:
    shutil.copy2(str(MODEL_PATH), DRIVE_MODEL_PATH)
    print(f'Model synced to Drive: {DRIVE_MODEL_PATH}')

print(f'\nModel saved: {MODEL_PATH}')

### Training History

In [ ]:
import json

history_path = RESULTS_DIR / 'training_results' / 'training_history.json'
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for stage in ['stage1', 'stage2']:
        h = history.get(stage, {})
        if 'accuracy' in h:
            ax1.plot(h['accuracy'], label=f'{stage} train')
        if 'val_accuracy' in h:
            ax1.plot(h['val_accuracy'], label=f'{stage} val')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True)
    for stage in ['stage1', 'stage2']:
        h = history.get(stage, {})
        if 'loss' in h:
            ax2.plot(h['loss'], label=f'{stage} train')
        if 'val_loss' in h:
            ax2.plot(h['val_loss'], label=f'{stage} val')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print(f'No training history at {history_path}')

## 9. Evaluation on Test Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import seaborn as sns

IMG_SIZE = (224, 224)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='int',
    image_size=IMG_SIZE,
    batch_size=8,
    shuffle=False,
)
class_names = test_ds.class_names
print(f'Test classes: {class_names}')

model = tf.keras.models.load_model(str(MODEL_PATH))
print(f'Model loaded: {MODEL_PATH}')

y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f'\nTest Accuracy:  {acc:.4f}')
print(f'Test Precision: {precision:.4f}')
print(f'Test Recall:    {recall:.4f}')
print(f'Test F1-score:  {f1:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, len(class_names)), max(6, len(class_names) * 0.7)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix -- Test Set')
plt.tight_layout()
plt.show()

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
per_class_f1 = [report[c]['f1-score'] for c in class_names]

plt.figure(figsize=(10, 5))
plt.bar(range(len(class_names)), per_class_f1, color='coral')
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
plt.ylabel('F1-score')
plt.title('Per-Class F1-Score on Test Set')
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

## 10. Health-Label Rollup

In [ ]:
y_true_health = [HEALTH_LABELS.get(class_names[i], 'unknown') for i in y_true]
y_pred_health = [HEALTH_LABELS.get(class_names[i], 'unknown') for i in y_pred]

health_labels = ['healthy', 'unhealthy', 'not_food']
print('Health-level classification report:')
print(classification_report(y_true_health, y_pred_health, labels=health_labels, zero_division=0))

cm_health = confusion_matrix(y_true_health, y_pred_health, labels=health_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_health, annot=True, fmt='d', cmap='Oranges',
            xticklabels=health_labels, yticklabels=health_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Health-Level Confusion Matrix')
plt.tight_layout()
plt.show()

## 11. Inference Demo

In [ ]:
import cv2
from PIL import Image

UNHEALTHY_CLASSES = {k for k, v in HEALTH_LABELS.items() if v == 'unhealthy'}

def predict_single_image(model, img_path, class_names):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224)).astype('float32')
    img_batch = np.expand_dims(img_resized, axis=0)

    preds = model.predict(img_batch, verbose=0)[0]
    top_indices = np.argsort(preds)[::-1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    predicted_class = class_names[top_indices[0]]
    health = HEALTH_LABELS.get(predicted_class, 'unknown')
    ax1.imshow(img_rgb)
    ax1.set_title(f'Prediction: {predicted_class} ({preds[top_indices[0]]:.1%}) [{health}]')
    ax1.axis('off')
    ax2.barh(range(len(class_names)), [preds[i] for i in range(len(class_names))], color='steelblue')
    ax2.set_yticks(range(len(class_names)))
    ax2.set_yticklabels(class_names)
    ax2.set_xlabel('Confidence')
    ax2.set_title('Class Probabilities')
    ax2.set_xlim(0, 1)
    plt.tight_layout()
    plt.show()
    print(f'Health group: {health}')

test_classes = [d for d in TEST_DIR.iterdir() if d.is_dir()]
if test_classes:
    sample_class = test_classes[0]
    sample_imgs = list(sample_class.glob('*.jpg'))[:1] or list(sample_class.glob('*.png'))[:1]
    if sample_imgs:
        predict_single_image(model, sample_imgs[0], class_names)
else:
    print('No test images found.')

## 12. Export to TFLite

In [ ]:
from convert_to_tflite import convert

TFLITE_PATH = MODEL_PATH.with_suffix('.tflite')

if MODEL_PATH.exists():
    convert(MODEL_PATH, TFLITE_PATH, quantize=True)
    print(f'\nTFLite model: {TFLITE_PATH}')
    print(f'Size: {TFLITE_PATH.stat().st_size / 1024:.0f} KB')

    if DRIVE_CHECKPOINT_DIR:
        drive_tflite = os.path.join(DRIVE_CHECKPOINT_DIR, TFLITE_PATH.name)
        shutil.copy2(str(TFLITE_PATH), drive_tflite)
        print(f'TFLite synced to Drive: {drive_tflite}')
else:
    print('No trained model found. Run training first.')

## 13. Upload Model to HuggingFace

In [ ]:
# Upload trained model after training completes
SKIP_MODEL_UPLOAD = False

if not SKIP_MODEL_UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi, create_repo

    HF_MODEL_REPO = 'maia2000/efficientnet-food-classifier'
    api = HfApi(token=HF_TOKEN)
    try:
        create_repo(HF_MODEL_REPO, token=HF_TOKEN, exist_ok=True)
    except Exception as e:
        print(f'Repo creation: {e}')

    files_to_upload = []
    if MODEL_PATH.exists():
        files_to_upload.append((str(MODEL_PATH), MODEL_PATH.name))
    if TFLITE_PATH.exists():
        files_to_upload.append((str(TFLITE_PATH), TFLITE_PATH.name))

    for local_path, name in files_to_upload:
        print(f'Uploading {name}...')
        api.upload_file(path_or_fileobj=local_path, path_in_repo=name, repo_id=HF_MODEL_REPO, token=HF_TOKEN)
        print(f'  Uploaded: {name}')
    print(f'\nDone! https://huggingface.co/{HF_MODEL_REPO}')
elif not HF_TOKEN:
    print('[SKIP] No HF_TOKEN -- add it to Colab Secrets')
else:
    print('[SKIP] Model upload')

## Model Summary

In [ ]:
model.summary()

total_params = model.count_params()
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
non_trainable = total_params - trainable

print(f'\nTotal params:         {total_params:,}')
print(f'Trainable params:     {trainable:,}')
print(f'Non-trainable params: {non_trainable:,}')
if MODEL_PATH.exists():
    print(f'Model file size:      {MODEL_PATH.stat().st_size / (1024 * 1024):.1f} MB')
if TFLITE_PATH.exists():
    print(f'TFLite file size:     {TFLITE_PATH.stat().st_size / 1024:.0f} KB')